# Isoprene Jacobians

---
Last modified: March 6th, 2025 (JY)

This notebook calculates Jacobians for isoprene vertical profiles using FiniteDiff.jl. This notebook is coded in Julia.

In [24]:
# Activate Julia packages
using Pkg;
Pkg.activate("../"); # Activates the vSmartMOM.jl Project.toml file
Pkg.instantiate();

# Using local module, since the current version of vSmartMOM does not support isoprene (not a HITRAN line list species)
include("../src/vSmartMOM.jl");
include("helper_functions.jl");

# Other imports
using Revise, PlotlyBase, Plots, LinearAlgebra, NCDatasets, Format, LaTeXStrings, .vSmartMOM;
using FiniteDiff;

default(fontfamily="Computer Modern");

  Activating project at `~/Documents/Radiative_Transfer/vSmartMOM.jl`


### We now need to specify parameters for the model.

I have downloaded 10 days (2019-07-01 to 2019-07-10) in the MERRA2 folder. You will need to manually download more days if you would like different conditions.

In [3]:
# Parameters for atmospheric profile
date = "20190701";
startTime = "06z";
lat = 2.;
lon = 100.;

merra2_profile = retrieve_merra2_conditions(date, startTime, lat, lon);
merra2_profile = reduce_profile(4, merra2_profile);

pressures = merra2_profile.p_levels ./ 100; # Convert to hPa
pressures_midpoints = merra2_profile.p ./ 100; # Convert to hPa
temperatures = merra2_profile.T;
specific_humidity = merra2_profile.q .* 1000; # Convert from kg/kg (in MERRA2) to g/kg;

ds["T"] = T (576 × 361 × 72 × 4)
  Datatype:    Union{Missing, Float32} (Float32)
  Dimensions:  lon × lat × lev × time
  Attributes:
   long_name            = Air temperature
   units                = K
   _FillValue           = 1.0e15
   missing_value        = 1.0e15
   fmissing_value       = 1.0e15
   scale_factor         = 1.0
   add_offset           = 0.0
   standard_name        = air_temperature
   vmax                 = 1.0e15
   vmin                 = -1.0e15
   valid_range          = Float32[-1.0f15, 1.0f15]



In [4]:
current_profile = specific_humidity ./ 1000 .* 1e-7; # Get current isoprene vertical profile
total_column = merra2_profile.vcd_dry .* current_profile; # Multiply by vertical column density
total_column = sum(total_column) # Sum for the total column
println("The total column is $(total_column) molec cm-2")

mean_vmr = total_column / sum(merra2_profile.vcd_dry)
println("The equivalent constant VMR is $(mean_vmr * 1e9) ppbv")

current_profile_x5 = specific_humidity ./ 1000 .* 1e-7 .* 5; # Get current isoprene vertical profile
total_column_x5 = merra2_profile.vcd_dry .* current_profile_x5; # Multiply by vertical column density
total_column_x5 = sum(total_column_x5) # Sum for the total column
println("The total column is $(total_column_x5) molec cm-2")

mean_vmr_x5 = total_column_x5 / sum(merra2_profile.vcd_dry)
println("The equivalent constant VMR is $(mean_vmr_x5 * 1e9) ppbv")

The total column is 1.0713622913766074e16 molec cm-2
The equivalent constant VMR is 0.5095802019910322 ppbv
The total column is 5.356811456883037e16 molec cm-2
The equivalent constant VMR is 2.547901009955161 ppbv


In [5]:
import InstrumentOperator: FTSInstrument, create_instrument_kernel, FixedKernelInstrument, conv_spectra

ν_min = 890.
ν_max = 910.
Δν = 0.01
ν = ν_min:Δν:ν_max

x = -5:Δν:5
fov = 16.8 #mrads
mopd = 0.8 #cm
FTS = FTSInstrument(0.8, fov * 1e-3, 0.)
unapodized_fts = create_instrument_kernel(FTS, x, 900.0)

p = plot(x, unapodized_fts.parent,label="CrIS Unapodized FTS Lineshape",lw=2, dpi = 300, title = "CrIS Unapodized FTS Instrument Line Shape \n(FOV = 16.8 mrads, MOPD = 0.8 cm)")
ylabel!("Channel Response Function")
xlabel!("σ (cm^-1)")
# savefig(p, "Figures/unapodized_instrument_kernel.png")

margin = 0
sampling = 0.625
cris_instrument_kernel = FixedKernelInstrument(unapodized_fts, collect(ν_min+margin:sampling:ν_max-margin))

┌ Info: Center of OffsetArrays is 
│   minimum(abs.(grid_x)) = 0.0
└ @ InstrumentOperator /Users/jamesyoon/.julia/packages/InstrumentOperator/JJWCH/src/prepare_ils.jl:39


(Θ, β) = (0.12700800000000004, 0.0)


FixedKernelInstrument{Float64}([-0.0001938027449118005, -0.00022515990131574464, -0.0002560745521885616, -0.0002864668905839412, -0.0003162580989748737, -0.0003453705508071965, -0.00037372800988458884, -0.00040125582708002165, -0.0004278811338732687, -0.00045353303222005073  …  -0.00011292824838757243, -9.056541516724512e-5, -7.058344327000462e-5, -5.30230478386403e-5, -3.791869420702852e-5, -2.529853680148194e-5, -1.5184374041206004e-5, -7.5916192866803565e-6, -2.5292878448332848e-6, -1.463672932855431e-18], [890.0, 890.625, 891.25, 891.875, 892.5, 893.125, 893.75, 894.375, 895.0, 895.625  …  904.375, 905.0, 905.625, 906.25, 906.875, 907.5, 908.125, 908.75, 909.375, 910.0])

### Running the Jacobian

In [6]:
parameters = vSmartMOM.CoreRT.parameters_from_yaml("yaml_files/scattering_studies.yaml");

# Prior State Vector
x = ones(length(specific_humidity)) # ./ 1000 * 28.96/18.02 .* 1000

# Runner is used to wrap the vSmartMOM model
function runner(x, vza_index = 1, parameters = parameters)
    x = x / 1e9 # Convert from ppb to mole fractions
    parameters.T = temperatures;
    parameters.p = pressures;
    parameters.q = specific_humidity;
    parameters.absorption_params.vmr["H2O"] = 0;
    parameters.absorption_params.vmr["ISOP"] = x;

    model = vSmartMOM.CoreRT.model_from_parameters(parameters); # Create model from the YAML file
    model_output = vSmartMOM.CoreRT.rt_run(model); # Run the model
    model_output = conv_spectra(cris_instrument_kernel, ν, model_output[1][vza_index,1,:]);
    return model_output;
end;

In [7]:
jacobian = FiniteDiff.finite_difference_jacobian(runner, x)

┌ Info: Warning, make sure that the VMR is interpolated correctly! Right now, it might be tricky
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM.jl/src/CoreRT/tools/atmo_prof.jl:79


(params.absorption_params.molecules[i_band])[molec_i] = "CO2"
Computing profile for CO2 with vmr [0.000385, 0.000388274682120417, 0.0003916634513431471, 0.000395] for band #1


Progress: 100%|█████████████████████████████████████████| Time: 0:00:01


(params.absorption_params.molecules[i_band])[molec_i] = "O2"
Computing profile for O2 with vmr 0.21 for band #1
(params.absorption_params.molecules[i_band])[molec_i] = "ISOP"
Computing profile for ISOP with vmr [1.0e-9, 1.0e-9, 1.0e-9, 1.0e-9] for band #1


Progress: 100%|█████████████████████████████████████████| Time: 0:00:02


...

┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=1
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM.jl/src/CoreRT/tools/postprocessing_vza.jl:63


Fourier Moment: 2/2


┌ Info: Radiance Δ range in %: from 0.0 to 0.0 for m=2
└ @ Main.vSmartMOM.CoreRT /Users/jamesyoon/Documents/Radiative_Transfer/vSmartMOM.jl/src/CoreRT/tools/postprocessing_vza.jl:63


───────────────────────────────────────────────────────────────────────────────────────
                                              Time                    Allocations      
                                     ───────────────────────   ────────────────────────
          Tot / % measured:               11.8s /  53.5%           13.0GiB /  79.9%    

Section                      ncalls     time    %tot     avg     alloc    %tot      avg
───────────────────────────────────────────────────────────────────────────────────────
Absorption Coeff                  4    3.50s   55.5%   876ms   2.79GiB   26.9%   715MiB
RT Kernel                        12    2.61s   41.4%   218ms   6.91GiB   66.6%   590MiB
  doubling                       12    2.30s   36.4%   192ms   6.19GiB   59.6%   529MiB
    Batch Inv Doubling          132    1.12s   17.7%  8.45ms   2.61GiB   25.1%  20.2MiB
  interaction                     9    209ms    3.3%  23.2ms    729MiB    6.9%  81.0MiB
    interaction inv1 bla       

33×4 Matrix{Float64}:
 -5.21764e-5   -4.40553e-5   -2.09333e-5  -8.08388e-7
 -6.50352e-5   -4.28734e-5   -1.70832e-5   1.24704e-6
 -6.91768e-5   -4.62681e-5   -1.96733e-5   1.00024e-6
 -9.09939e-5   -6.03329e-5   -2.76482e-5   1.39698e-8
 -0.000133414  -8.91034e-5   -3.93055e-5   1.38115e-6
 -0.00020469   -0.000136496  -5.85746e-5   3.19444e-7
 -0.000281467  -0.000184429  -7.76025e-5   1.83191e-6
 -0.000106307  -7.32848e-5   -3.17516e-5   2.07126e-6
 -5.39599e-5   -3.58364e-5   -1.57822e-5   1.90921e-7
 -5.13885e-5   -3.52263e-5   -1.56714e-5   1.65217e-6
  ⋮                                       
 -0.000125724  -8.39503e-5   -3.63998e-5   5.95115e-7
 -0.000144046  -9.74257e-5   -4.22625e-5   5.69969e-7
 -0.000240221  -0.000158986  -6.6869e-5    1.85799e-6
 -0.000102518  -6.9554e-5    -3.01348e-5   1.58604e-6
 -5.58197e-5   -3.72725e-5   -1.56565e-5   6.70552e-8
 -5.85904e-5   -4.0804e-5    -1.74316e-5   4.74975e-7
 -6.10901e-5   -4.02862e-5   -1.73897e-5  -9.31323e-10
 -6.09411e-5   -

In [30]:
p = plot(cris_instrument_kernel.ν_out, jacobian[:,1], label = "$(pressures[1]) hPa", lw = 2, dpi = 600)
plot!(cris_instrument_kernel.ν_out, jacobian[:,2], label = "$(round(pressures[2])) hPa", lw = 2)
plot!(cris_instrument_kernel.ν_out, jacobian[:,3], label = "$(round(pressures[3])) hPa", lw = 2)
plot!(cris_instrument_kernel.ν_out, jacobian[:,4], label = "$(round(pressures[4])) hPa", lw = 2)

title!("Isoprene Jacobians @ 4 Altitudes (step = 1 ppbv)")
xlabel!(L"Wavenumber $\mathrm{[cm^{-1}]}$")
ylabel!(L"Jacobian of Radiance $\mathrm{\left( \frac{dR}{dx_i} \right)}$")
savefig(p, "figures/jacobian.png");